# MusicSage Separator

Complex 2D U-Net for music source separation

- Bandwidth: 44.1 kHz
- Multi-resolution STFT input (5 resolutions)
- Complex masking (8-channel output: real+imag for 4 sources)
- Gradient accumulation + mixed precision
- ~70M parameters, fits in 15GB VRAM

In [ ]:
import os
import sys
import math
from functools import partial

import numpy as np
import soundfile as sf
import tensorflow as tf
from tensorflow.keras import layers, models

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

EPS = 1e-8

---
## 1. Multi-resolution STFT config

In [ ]:
MR_STFT_CONFIGS = [
    (4096, 1024),  # reference — ISTFT from this resolution
    (8192, 2048),  # high frequency resolution
    (2048, 512),   # mid frequency resolution
    (1024, 256),   # high time resolution
    (256, 64),     # very high time resolution
]
CONV_SIZE = 32  # must match 2^n_levels
CHUNK_SEC = 10
SAMPLE_RATE = 44100

def _make_window_fn(frame_length):
    window = tf.sqrt(tf.signal.hann_window(frame_length, periodic=True))
    def _fn(fl, dtype):
        return tf.cast(window, dtype)
    return _fn

def pad_feature(x):
    h, w = tf.shape(x)[0], tf.shape(x)[1]
    pad_h = (CONV_SIZE - h % CONV_SIZE) % CONV_SIZE
    pad_w = (CONV_SIZE - w % CONV_SIZE) % CONV_SIZE
    paddings = [[0, pad_h], [0, pad_w]]
    ndims = x.shape.rank
    if ndims is not None and ndims > 2:
        paddings.extend([[0, 0]] * (ndims - 2))
    return tf.pad(x, paddings)

---
## 2. Model: Complex 2D U-Net

In [ ]:
def conv_block(x, filters, kernel_size=3):
    shortcut = x
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.GroupNormalization(groups=min(filters // 8, 32))(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.GroupNormalization(groups=min(filters // 8, 32))(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same')(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x


class ECABlock(layers.Layer):
    """Efficient Channel Attention — 1D conv along channels."""
    def __init__(self, gamma=2, b=1, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.b = b

    def build(self, input_shape):
        channels = input_shape[-1]
        k = max(int(abs(math.log2(channels) / self.gamma + self.b / self.gamma)), 3)
        k = k if k % 2 == 1 else k + 1
        self.gap = layers.GlobalAveragePooling2D()
        self.conv = layers.Conv1D(1, k, padding='same', use_bias=False)

    def call(self, x):
        gap = self.gap(x)[..., tf.newaxis]
        gap = self.conv(gap)
        gap = tf.sigmoid(gap)
        gap = tf.transpose(gap, [0, 2, 1])
        gap = gap[:, tf.newaxis, :, :]
        return layers.multiply([x, gap])


class MHABottleneck(layers.Layer):
    """Multi-head self-attention at bottleneck (small spatial dims)."""
    def __init__(self, n_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.n_heads = n_heads

    def build(self, input_shape):
        c = input_shape[-1]
        self.mha = layers.MultiHeadAttention(
            num_heads=self.n_heads, key_dim=max(c // self.n_heads // 2, 16)
        )
        self.ln = layers.LayerNormalization()

    def call(self, x):
        b, h, w, c = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], x.shape[-1]
        x_t = tf.reshape(x, [b, h * w, c])
        attn_out = self.mha(x_t, x_t)
        attn_out = tf.reshape(attn_out, [b, h, w, c])
        return self.ln(x + attn_out)


def create_unet(
    input_shape=(None, None, 1),
    n_filters=32,
    n_levels=5,
    n_outputs=8,
    dropout_rate=0.2,
    use_eca=True,
    bottleneck_attention=True,
    n_input_channels=None,
):
    if n_input_channels is not None:
        input_shape = (input_shape[0], input_shape[1], n_input_channels)
    inputs = layers.Input(shape=input_shape)

    x = inputs
    skips = []
    filter_list = [n_filters * (2 ** i) for i in range(n_levels)]

    for filters in filter_list:
        x = conv_block(x, filters)
        x = conv_block(x, filters)
        if use_eca:
            x = ECABlock()(x)
        skips.append(x)
        x = layers.AveragePooling2D((2, 2))(x)

    bottleneck_filters = filter_list[-1] * 2
    x = conv_block(x, bottleneck_filters)
    x = conv_block(x, bottleneck_filters)
    if use_eca:
        x = ECABlock()(x)
    if bottleneck_attention:
        x = MHABottleneck(n_heads=4)(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    for i in range(n_levels - 1, -1, -1):
        filters = filter_list[i]
        x = layers.UpSampling2D((2, 2))(x)
        x = layers.Conv2D(filters, 3, padding='same')(x)
        x = layers.GroupNormalization(groups=min(filters // 8, 32))(x)
        x = layers.Activation('relu')(x)
        x = layers.concatenate([x, skips[i]])
        x = conv_block(x, filters)
        x = conv_block(x, filters)
        if use_eca and i > 0:
            x = ECABlock()(x)
        if dropout_rate > 0 and i > 0:
            x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Conv2D(n_outputs, 1, padding='same', dtype='float32')(x)
    return models.Model(inputs, outputs)

In [ ]:
# Build and inspect
n_channels = len(MR_STFT_CONFIGS)
unet = create_unet(input_shape=(None, None, n_channels))
unet.build((1, 448, 2080, n_channels))
unet.summary()
print(f'\nTotal parameters: {unet.count_params():,}')

---
## 3. Loss: ComplexSeparatorLoss

Three components:
1. **Magnitude mask loss** — L1(|complex_mask| - target_mag_mask)
2. **Complex STFT loss** — L1(|pred_src_stft - target_src_stft|) — phase-aware
3. **Multi-resolution spectral loss** — L1 on pooled magnitude spectra

In [ ]:
class ComplexSeparatorLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=1.0, beta=1.0, gamma=0.1, scales=(1, 2, 4)):
        super().__init__(name='complex_separator_loss')
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.scales = scales

    def call(self, y_true, y_pred):
        target_mag_masks = y_true[..., :4]
        tgt_r = y_true[..., 4:8]
        tgt_i = y_true[..., 8:12]
        mix_r = y_true[..., 12:13]
        mix_i = y_true[..., 13:14]

        pred_r = y_pred[..., :4]
        pred_i = y_pred[..., 4:8]

        pred_mag = tf.sqrt(pred_r ** 2 + pred_i ** 2 + EPS)
        mag_loss = tf.reduce_mean(tf.abs(pred_mag - target_mag_masks))

        src_r = pred_r * mix_r - pred_i * mix_i
        src_i = pred_r * mix_i + pred_i * mix_r

        stft_mag = tf.sqrt((src_r - tgt_r) ** 2 + (src_i - tgt_i) ** 2 + EPS)
        stft_loss = tf.reduce_mean(stft_mag)

        spec_loss = 0.0
        for s in self.scales:
            pred_src_mag = tf.sqrt(src_r ** 2 + src_i ** 2 + EPS)
            tgt_src_mag = tf.sqrt(tgt_r ** 2 + tgt_i ** 2 + EPS)
            if s > 1:
                pred_src_mag = tf.nn.avg_pool2d(pred_src_mag, s, s, 'SAME')
                tgt_src_mag = tf.nn.avg_pool2d(tgt_src_mag, s, s, 'SAME')
            spec_loss += tf.reduce_mean(tf.abs(pred_src_mag - tgt_src_mag))
        spec_loss /= len(self.scales)

        return self.alpha * mag_loss + self.beta * stft_loss + self.gamma * spec_loss


def mask_mae(y_true, y_pred):
    pred_mag = tf.sqrt(y_pred[..., :4] ** 2 + y_pred[..., 4:8] ** 2 + 1e-8)
    return tf.reduce_mean(tf.abs(pred_mag - y_true[..., :4]))

---
## 4. Data pipeline

In [ ]:
FEATURE_DESCRIPTION = {
    'length': tf.io.FixedLenFeature([], tf.int64),
    'mix': tf.io.FixedLenFeature([], tf.string),
    'drums': tf.io.FixedLenFeature([], tf.string),
    'bass': tf.io.FixedLenFeature([], tf.string),
    'other': tf.io.FixedLenFeature([], tf.string),
    'vocals': tf.io.FixedLenFeature([], tf.string),
}


def parse_tfrecord(example_proto):
    example = tf.io.parse_single_example(example_proto, FEATURE_DESCRIPTION)
    length = tf.cast(example['length'], tf.int32)
    def decode(key):
        return tf.reshape(tf.io.decode_raw(example[key], tf.float32), [length])
    return {
        'mix': decode('mix'), 'drums': decode('drums'),
        'bass': decode('bass'), 'other': decode('other'),
        'vocals': decode('vocals'),
    }


def preprocess(example, configs=None, augment=False):
    if configs is None:
        configs = MR_STFT_CONFIGS

    tracks = tf.stack([
        example['mix'], example['drums'], example['bass'],
        example['other'], example['vocals'],
    ])
    mix = example['mix']

    if augment:
        gain = tf.random.uniform([], 0.8, 1.25)
        tracks = tracks * gain
        mix = mix * gain

    ref_fl, ref_fs = configs[0]
    ref_wfn = _make_window_fn(ref_fl)
    ref_stft = tf.signal.stft(
        tracks, frame_length=ref_fl, frame_step=ref_fs,
        fft_length=ref_fl, window_fn=ref_wfn
    )
    ref_mag = tf.abs(ref_stft)
    ref_T, ref_F = tf.shape(ref_mag)[1], tf.shape(ref_mag)[2]

    # Multi-resolution STFT for mix → input channels
    mix_channels = []
    for i, (fl, fs) in enumerate(configs):
        wfn = _make_window_fn(fl)
        s = tf.signal.stft(mix, frame_length=fl, frame_step=fs,
                           fft_length=fl, window_fn=wfn)
        log_mag = tf.math.log1p(tf.abs(s))
        if i == 0:
            mix_channels.append(log_mag)
        else:
            log_mag = tf.image.resize(
                log_mag[tf.newaxis, ..., tf.newaxis], [ref_T, ref_F]
            )
            mix_channels.append(log_mag[0, ..., 0])
    mix_input = tf.stack(mix_channels, axis=-1)

    # Magnitude masks (stabilizes training)
    masks = ref_mag[1:] / (ref_mag[0] + EPS)
    masks = tf.clip_by_value(masks, 0.0, 1.0)
    masks = tf.transpose(masks, [1, 2, 0])

    # Complex STFT data for loss
    source_stft = ref_stft[1:]
    source_real = tf.transpose(tf.math.real(source_stft), [1, 2, 0])
    source_imag = tf.transpose(tf.math.imag(source_stft), [1, 2, 0])
    mix_real = tf.math.real(ref_stft[0])[..., tf.newaxis]
    mix_imag = tf.math.imag(ref_stft[0])[..., tf.newaxis]

    # Pad to CONV_SIZE alignment
    mix_input = pad_feature(mix_input)
    masks = pad_feature(masks)
    source_real = pad_feature(source_real)
    source_imag = pad_feature(source_imag)
    mix_real = pad_feature(mix_real)
    mix_imag = pad_feature(mix_imag)

    target = tf.concat([
        masks, source_real, source_imag, mix_real, mix_imag
    ], axis=-1)
    return mix_input, target


class Normalizer:
    def __init__(self, stats_file=None):
        if stats_file is None:
            self.mean = self.std = None
        else:
            stats = np.load(stats_file)
            mean, std = stats['mean'], stats['std']
            if mean.ndim == 0:
                self.mean = tf.constant(float(mean), dtype=tf.float32)
                self.std = tf.constant(float(std), dtype=tf.float32)
            else:
                self.mean = tf.constant(mean, dtype=tf.float32)
                self.std = tf.constant(std, dtype=tf.float32)

    def __call__(self, x, y):
        if self.mean is None:
            return x, y
        x = (x - self.mean) / (self.std + 1e-8)
        return x, y


def create_dataset(
    tfrecord_dir, batch_size, stats_file=None, shuffle=True,
    shuffle_buffer=128, augment=False,
):
    files = tf.data.Dataset.list_files(
        os.path.join(tfrecord_dir, '*.tfrecord'), shuffle=shuffle
    )
    ds = files.interleave(
        lambda x: tf.data.TFRecordDataset(x, compression_type='GZIP'),
        cycle_length=2, block_length=1, num_parallel_calls=2,
        deterministic=not shuffle
    )
    if shuffle:
        ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
    ds = ds.map(parse_tfrecord, num_parallel_calls=2)
    ds = ds.map(
        partial(preprocess, augment=augment), num_parallel_calls=2
    )
    normalizer = Normalizer(stats_file)
    ds = ds.map(normalizer, num_parallel_calls=2)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(2)
    return ds

---
## 5. Training setup

In [ ]:
class EMACallback(tf.keras.callbacks.Callback):
    """Exponential Moving Average of weights."""
    def __init__(self, decay=0.9999):
        super().__init__()
        self.decay = decay
        self.ema_weights = None

    def on_train_begin(self, logs=None):
        self.ema_weights = [
            tf.Variable(tf.identity(w), trainable=False)
            for w in self.model.trainable_weights
        ]

    def on_batch_end(self, batch, logs=None):
        for i, w in enumerate(self.model.trainable_weights):
            self.ema_weights[i].assign(
                self.decay * self.ema_weights[i]
                + (1 - self.decay) * w
            )

    def on_train_end(self, logs=None):
        for w, ema in zip(self.model.trainable_weights, self.ema_weights):
            w.assign(ema)

In [ ]:
# Enable mixed precision
tf.keras.mixed_precision.set_global_policy('mixed_float16')

BATCH_SIZE = 8
EPOCHS = 100
LEARNING_RATE = 2e-4

TFRECORD_DIR = 'musdb18/tfrecord'
STATS_FILE = 'musdb18/stats.npz'

train_ds = create_dataset(
    tfrecord_dir=TFRECORD_DIR,
    batch_size=BATCH_SIZE,
    stats_file=STATS_FILE if os.path.exists(STATS_FILE) else None,
    shuffle=True,
    augment=True,
)

# Optional validation set
val_ds = None
if os.path.isdir('musdb18/tfrecord_test'):
    val_ds = create_dataset(
        tfrecord_dir='musdb18/tfrecord_test',
        batch_size=BATCH_SIZE,
        stats_file=STATS_FILE if os.path.exists(STATS_FILE) else None,
        shuffle=False,
    )

for x, y in train_ds.take(1):
    print('Input:', x.shape, x.dtype)
    print('Target:', y.shape, y.dtype)

In [ ]:
# Build model
model = create_unet(input_shape=(None, None, n_channels))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=ComplexSeparatorLoss(alpha=1.0, beta=1.0, gamma=0.1),
    metrics=[mask_mae],
)

callbacks = [
    EMACallback(),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss' if val_ds else 'loss',
        factor=0.5, patience=5, min_lr=1e-7
    ),
]

if val_ds is not None:
    callbacks.extend([
        tf.keras.callbacks.ModelCheckpoint(
            'checkpoint.keras', save_best_only=True, monitor='val_loss'
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=15, restore_best_weights=True
        ),
    ])

In [ ]:
# ────────────────────────────────────────────────────────────────
# TRAIN
# ────────────────────────────────────────────────────────────────
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
)

model.save('separator.keras')
print('Done. Saved separator.keras')

---
## 6. Inference

Complex masking: `source_stft = (mask_r + 1j*mask_i) × mix_stft` → ISTFT

In [ ]:
def pad_spectrogram(spec):
    h, w = spec.shape[:2]
    pad_h = (CONV_SIZE - h % CONV_SIZE) % CONV_SIZE
    pad_w = (CONV_SIZE - w % CONV_SIZE) % CONV_SIZE
    if pad_h == 0 and pad_w == 0:
        return spec
    pad_width = [(0, pad_h), (0, pad_w)]
    for _ in range(spec.ndim - 2):
        pad_width.append((0, 0))
    return np.pad(spec, pad_width, mode='constant')


def load_audio(path, sr=44100):
    audio, orig_sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if orig_sr != sr:
        audio = tf.signal.resample(audio, orig_sr, sr).numpy()
    return audio


def separate(audio, model, configs=None, stats_file=None):
    if configs is None:
        configs = MR_STFT_CONFIGS

    if stats_file is not None:
        stats = np.load(stats_file)
        norm_mean = stats['mean'].astype(np.float32)
        norm_std = stats['std'].astype(np.float32)
    else:
        norm_mean = norm_std = None

    window_fns = [_make_window_fn(fl) for fl, _ in configs]

    ref_fl, ref_fs = configs[0]
    ref_wfn = window_fns[0]

    ref_stft = tf.signal.stft(
        audio, frame_length=ref_fl, frame_step=ref_fs,
        fft_length=ref_fl, window_fn=ref_wfn
    ).numpy()
    T, F = ref_stft.shape

    # Multi-resolution input
    channels = []
    for i, ((fl, fs), wfn) in enumerate(zip(configs, window_fns)):
        s = tf.signal.stft(audio, frame_length=fl, frame_step=fs,
                           fft_length=fl, window_fn=wfn)
        log_mag = np.log1p(np.abs(s.numpy()))
        if i == 0:
            channels.append(log_mag)
        else:
            log_mag = tf.image.resize(
                log_mag[np.newaxis, :, :, np.newaxis], [T, F]
            ).numpy()[0, :, :, 0]
            channels.append(log_mag)

    spec = np.stack(channels, axis=-1).astype(np.float32)
    if norm_mean is not None:
        spec = (spec - norm_mean) / (norm_std + 1e-8)

    padded_spec = pad_spectrogram(spec)
    T_pad, F_pad = padded_spec.shape[:2]

    # Overlap-Add with hanning window
    chunk_frames = 448
    chunk_hop = chunk_frames // 2

    mask_accum = np.zeros((T_pad, F_pad, 8), dtype=np.float64)
    weight_accum = np.zeros(T_pad, dtype=np.float64)
    ola_window = np.hanning(chunk_frames)

    for start in range(0, T_pad, chunk_hop):
        end = start + chunk_frames
        chunk = padded_spec[start:end, :, :]
        pad = 0
        if chunk.shape[0] < chunk_frames:
            pad = chunk_frames - chunk.shape[0]
            chunk = np.pad(chunk, [(0, pad), (0, 0), (0, 0)])

        pred = model.predict(chunk[np.newaxis, ...], verbose=0)[0]
        if pad:
            pred = pred[:-pad]

        n = pred.shape[0]
        mask_accum[start:start+n] += pred * ola_window[:n, np.newaxis, np.newaxis]
        weight_accum[start:start+n] += ola_window[:n]

    mask_accum /= weight_accum[:, np.newaxis, np.newaxis] + 1e-8
    pred = mask_accum[:T, :F, :].astype(np.float32)

    mask_r = pred[:, :, :4]
    mask_i = pred[:, :, 4:]

    # ISTFT with complex masking
    sources = []
    for i in range(4):
        source_stft = (mask_r[:, :, i] + 1j * mask_i[:, :, i]) * ref_stft
        source_wav = tf.signal.inverse_stft(
            source_stft,
            frame_length=ref_fl, frame_step=ref_fs,
            fft_length=ref_fl, window_fn=ref_wfn,
        ).numpy()
        sources.append(source_wav[:len(audio)])

    return np.stack(sources, axis=-1)

In [ ]:
# Example inference
input_path = 'test_song.mp3'  # replace with your audio file
if os.path.exists(input_path):
    audio = load_audio(input_path)
    stems = separate(audio, model, stats_file=STATS_FILE)

    names = ['drums', 'bass', 'other', 'vocals']
    os.makedirs('separated', exist_ok=True)
    for i, name in enumerate(names):
        sf.write(f'separated/{name}.wav', stems[:, i], SAMPLE_RATE)
        print(f'Saved separated/{name}.wav')
else:
    print(f'{input_path} not found. Place an audio file to test inference.')

---
## 7. Compute SI-SDR (evaluation)

In [ ]:
def si_sdr(estimate, reference):
    estimate = estimate - np.mean(estimate)
    reference = reference - np.mean(reference)
    alpha = np.dot(estimate, reference) / (np.dot(reference, reference) + EPS)
    noise = estimate - alpha * reference
    sdr = 10 * np.log10(
        np.dot(alpha * reference, alpha * reference)
        / (np.dot(noise, noise) + EPS) + EPS
    )
    return float(sdr)

def evaluate(ref_dir, stems):
    names = ['drums', 'bass', 'other', 'vocals']
    sdrs = []
    for i, name in enumerate(names):
        ref_path = os.path.join(ref_dir, f'{name}.wav')
        if not os.path.exists(ref_path):
            continue
        ref = load_audio(ref_path)
        T = min(len(stems[:, i]), len(ref))
        sdr = si_sdr(stems[:T, i], ref[:T])
        sdrs.append(sdr)
        print(f'  {name:>6s}: {sdr:.2f} dB')
    if sdrs:
        print(f'  Average: {np.mean(sdrs):.2f} dB')